In [1]:
import os

In [2]:
%pwd

'c:\\Users\\Salih\\Desktop\\End-to-End-Machine-Learning-Project-with-MLFlow\\research'

In [3]:
os.chdir("../")

In [4]:
%pwd

'c:\\Users\\Salih\\Desktop\\End-to-End-Machine-Learning-Project-with-MLFlow'

In [5]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class ModelEvaluationConfig:
    root_dir: Path
    test_data_path: Path
    model_path: Path
    all_params: dict
    metric_file_name: Path
    target_column: str
    mlflow_uri: str

In [6]:
from mlProject.constants import *
from mlProject.utils.common import read_yaml, create_directories, save_json

class ConfigurationManager:
    def __init__(
        self, 
        config_filepath: Path = CONFIG_FILE_PATH, 
        params_filepath: Path = PARAMS_FILE_PATH,
        schema_filepath: Path = SCHEMA_FILE_PATH):
        
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        self.schema = read_yaml(schema_filepath)

        create_directories([self.config.artifacts_root])

    def get_model_evaluation_config(self) -> ModelEvaluationConfig:
        config = self.config.model_evaluation
        params = self.params.ElasticNet
        schema = self.schema.TARGET_COLUMN

        create_directories([config.root_dir])

        model_evaluation_config = ModelEvaluationConfig(
            root_dir = config.root_dir,
            test_data_path = config.test_data_path,
            model_path = config.model_path,
            all_params = params,
            metric_file_name = config.metric_file_name,
            target_column = schema.name,
            mlflow_uri = "https://dagshub.com/salih245/End-to-End-Machine-Learning-Project-with-MLFlow.mlflow"
        )

        return model_evaluation_config

In [7]:
import os
import pandas as pd
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from urllib.parse import urlparse
import mlflow
import mlflow.sklearn
import numpy as np
import joblib
import dagshub

c:\Users\Salih\Desktop\End-to-End-Machine-Learning-Project-with-MLFlow\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [8]:
class ModelEvaluation:
    def __init__(self, config: ModelEvaluationConfig):
        self.config = config

    def eval_metrics(self, actual, predicted):
        rmse = np.sqrt(mean_squared_error(actual, predicted))
        mae = mean_absolute_error(actual, predicted)
        r2 = r2_score(actual, predicted)

        return rmse, mae, r2
    
    def log_into_mlflow(self):
        
        test_data = pd.read_csv(self.config.test_data_path)
        model = joblib.load(self.config.model_path)

        test_x = test_data.drop([self.config.target_column], axis=1)
        test_y = test_data[[self.config.target_column]]

        dagshub.init(
            repo_owner="salih245",
            repo_name="End-to-End-Machine-Learning-Project-with-MLFlow",
            mlflow=True
        )

        with mlflow.start_run():
            predicted_qualities = model.predict(test_x)

            rmse, mae, r2 = self.eval_metrics(test_y, predicted_qualities)

            scores = {
                "rmse": rmse,
                "mae": mae,
                "r2": r2
            }

            save_json(path=Path(self.config.metric_file_name), data=scores)

            mlflow.log_params(self.config.all_params)
            mlflow.log_metric("rmse", rmse)
            mlflow.log_metric("mae", mae)
            mlflow.log_metric("r2", r2)

            mlflow.sklearn.log_model(model, "model")

In [9]:
try:
    config = ConfigurationManager()
    model_evaluation_config = config.get_model_evaluation_config()
    model_evaluation_config = ModelEvaluation(config=model_evaluation_config)
    model_evaluation_config.log_into_mlflow()
except Exception as e:
    raise e

[2026-05-18 14:35:13,269]: INFO: common: YAML file 'config\config.yaml' read successfully.]
[2026-05-18 14:35:13,273]: INFO: common: YAML file 'params.yaml' read successfully.]
[2026-05-18 14:35:13,275]: INFO: common: YAML file 'schema.yaml' read successfully.]
[2026-05-18 14:35:13,277]: INFO: common: Directory created: artifacts]
[2026-05-18 14:35:13,279]: INFO: common: Directory created: artifacts/model_evaluation]
[2026-05-18 14:35:13,896]: INFO: _client: HTTP Request: GET https://dagshub.com/api/v1/user "HTTP/1.1 200 OK"]


Accessing as salih245

[2026-05-18 14:35:13,906]: INFO: helpers: Accessing as salih245]
[2026-05-18 14:35:14,481]: INFO: _client: HTTP Request: GET https://dagshub.com/api/v1/repos/salih245/End-to-End-Machine-Learning-Project-with-MLFlow "HTTP/1.1 200 OK"]
[2026-05-18 14:35:15,035]: INFO: _client: HTTP Request: GET https://dagshub.com/api/v1/user "HTTP/1.1 200 OK"]


Initialized MLflow to track repo "salih245/End-to-End-Machine-Learning-Project-with-MLFlow"

[2026-05-18 14:35:15,039]: INFO: helpers: Initialized MLflow to track repo "salih245/End-to-End-Machine-Learning-Project-with-MLFlow"]


Repository salih245/End-to-End-Machine-Learning-Project-with-MLFlow initialized!

[2026-05-18 14:35:15,041]: INFO: helpers: Repository salih245/End-to-End-Machine-Learning-Project-with-MLFlow initialized!]
[2026-05-18 14:39:29,111]: INFO: common: Data saved to JSON file at: artifacts\model_evaluation\metrics.json]


2026/05/18 14:39:30 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/18 14:40:06 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run likeable-worm-106 at: https://dagshub.com/salih245/End-to-End-Machine-Learning-Project-with-MLFlow.mlflow/#/experiments/0/runs/7e21e7bcc611466189493c9aaa8d0d34
🧪 View experiment at: https://dagshub.com/salih245/End-to-End-Machine-Learning-Project-with-MLFlow.mlflow/#/experiments/0
